In [26]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
VALIDASI P-WAVE ARRIVAL + VISUALISASI LENGKAP (DENGAN DEBUGGING)
"""

import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from obspy.signal.trigger import recursive_sta_lta
import os
import random
from datetime import datetime
import pandas as pd

# =============================================
# 1. KONFIGURASI
# =============================================
JSON_PATH = '/Volumes/Extreme SSD/json_indonesia_juli_sesi_4/extracted_data_3c_4_SYNCED.json'
OUTPUT_DIR = '/Volumes/Extreme SSD/mcu_quake_output_replikasi_demo/validation_p_pick'

STA_WIN = 1.0
LTA_WIN = 10.0
SAMPLE_RATE = 100.0
N_GRID_SAMPLES = 48  # 4x4 grid x 3 (total 48 plot)
N_BEST_WORST = 3

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =============================================
# 2. FUNGSI BANTUAN
# =============================================

def calculate_sta_lta(trace_data, sr, sta_win, lta_win):
    """Hitung characteristic function STA/LTA."""
    sta_n = int(sta_win * sr)
    lta_n = int(lta_win * sr)
    if len(trace_data) < lta_n + sta_n:
        return np.zeros(len(trace_data))
    try:
        cft = recursive_sta_lta(trace_data, sta_n, lta_n)
        return cft
    except:
        return np.zeros(len(trace_data))

def plot_single_event_detail(ax, signal, noise, cft, sr, p_pick_sample, title, event_id, mag, network, station):
    """Plot detail satu event."""
    time_full = np.arange(len(np.concatenate([noise, signal]))) / sr - len(noise)/sr
    full_data = np.concatenate([noise, signal])
    
    ax.plot(time_full, full_data, color='#2c3e50', linewidth=0.8, alpha=0.7)
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='P-pick')
    ax.axvspan(0, 7, alpha=0.1, color='blue', label='Signal window (7s)')
    ax.axvspan(-7, 0, alpha=0.1, color='gray', label='Noise window (7s)')
    
    if len(cft) > 0:
        cft_norm = cft / np.max(cft) * 0.8 - 0.9
        ax.plot(time_full, cft_norm, color='green', linewidth=1.5, alpha=0.8, label='STA/LTA (scaled)')
    
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude')
    ax.set_title(f'{event_id}\nM{mag} | {network}.{station}', fontsize=10)
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-8, 8)
    y_max = max(1.2, np.max(np.abs(full_data)) * 1.1)
    ax.set_ylim(-y_max, y_max)

# =============================================
# 3. FUNGSI UTAMA
# =============================================

def validate_and_visualize(json_path):
    print("="*70)
    print("🔍 VALIDASI P-WAVE + VISUALISASI LENGKAP (DEBUG MODE)")
    print("="*70)
    
    # Cek file
    if not os.path.exists(json_path):
        print(f"❌ File tidak ditemukan: {json_path}")
        return
    
    # Load JSON
    print(f"📂 Memuat JSON: {json_path}")
    with open(json_path, 'r') as f:
        data = json.load(f)
    print(f"   Total entri: {len(data)}")
    
    if len(data) == 0:
        print("❌ JSON kosong. Tidak ada data.")
        return
    
    keys = list(data.keys())
    print(f"   Contoh 5 key pertama: {keys[:5]}")
    
    # ==========================================
    # A. EKSTRAKSI SEMUA DATA UNTUK STATISTIK
    # ==========================================
    print("📊 Mengumpulkan data statistik...")
    
    all_records = []
    travel_times = []
    networks = []
    stations = []
    magnitudes = []
    sta_lta_max = []
    event_ids = []
    
    # Untuk grid (sampling acak)
    grid_keys = random.sample(keys, min(N_GRID_SAMPLES, len(keys)))
    grid_data = []
    
    # Debug: periksa satu record pertama
    sample_key = keys[0]
    sample_record = data[sample_key]
    print(f"🔍 Struktur sample record (key: {sample_key}):")
    print(f"   Keys: {list(sample_record.keys())}")
    if 'metadata' in sample_record:
        print(f"   Metadata keys: {list(sample_record['metadata'].keys())}")
    print(f"   'type': {sample_record.get('type', 'MISSING')}")
    print(f"   'Z' length: {len(sample_record.get('Z', []))}")
    
    for key in keys:
        record = data[key]
        try:
            # Pastikan field yang diperlukan ada
            if 'Z' not in record or 'Z_noise' not in record:
                print(f"   ⚠️ Key {key}: missing Z or Z_noise, skip")
                continue
                
            signal = np.array(record['Z'], dtype=float)
            noise = np.array(record['Z_noise'], dtype=float)
            
            # Ambil metadata
            metadata = record.get('metadata', {})
            p_arrival_str = metadata.get('p_arrival', '')
            origin_str = metadata.get('origin_time', '')
            mag = metadata.get('magnitude', np.nan)
            network = metadata.get('network', 'UNK')
            station = metadata.get('station', 'UNK')
            label = record.get('type', 'unknown')
            
            if not p_arrival_str or not origin_str:
                travel_time = np.nan
            else:
                try:
                    p_time = datetime.fromisoformat(p_arrival_str.replace('Z', '+00:00'))
                    origin_time = datetime.fromisoformat(origin_str.replace('Z', '+00:00'))
                    travel_time = (p_time - origin_time).total_seconds()
                except Exception as e:
                    travel_time = np.nan
            
            # CFT
            full_trace = np.concatenate([noise, signal])
            cft = calculate_sta_lta(full_trace, SAMPLE_RATE, STA_WIN, LTA_WIN)
            max_cft = np.max(cft) if len(cft) > 0 else 0
            
            record_info = {
                'key': key,
                'signal': signal,
                'noise': noise,
                'cft': cft,
                'travel_time': travel_time,
                'magnitude': mag,
                'network': network,
                'station': station,
                'label': label,
                'max_cft': max_cft,
                'p_arrival': p_arrival_str,
                'origin': origin_str
            }
            
            all_records.append(record_info)
            
            # *** PERBAIKAN: Hanya simpan untuk statistik jika travel_time valid ***
            if not np.isnan(travel_time):
                travel_times.append(travel_time)
                networks.append(network)
                stations.append(station)
                magnitudes.append(mag)
                sta_lta_max.append(max_cft)
                event_ids.append(key)
            
            # Untuk grid
            if key in grid_keys:
                grid_data.append(record_info)
                
        except Exception as e:
            print(f"   ⚠️ Error pada key {key}: {e}")
            continue
    
    print(f"   ✅ Total data terekstrak: {len(all_records)}")
    print(f"   ✅ Grid data: {len(grid_data)} (dari {len(grid_keys)} yang diminta)")
    
    if len(all_records) == 0:
        print("❌ Tidak ada data valid untuk diproses. Periksa format JSON Anda.")
        return
    
    # ==========================================
    # B. DASHBOARD STATISTIK
    # ==========================================
    print("📈 Membuat dashboard statistik...")
    
    fig_dash = plt.figure(figsize=(16, 10))
    gs_dash = GridSpec(2, 3, figure=fig_dash, hspace=0.3, wspace=0.3)
    
    # B1. Histogram Travel Time
    ax1 = fig_dash.add_subplot(gs_dash[0, 0])
    if travel_times:
        arr = np.array(travel_times)
        # Filter hanya yang valid (tidak inf atau nan)
        arr = arr[~np.isnan(arr) & np.isfinite(arr)]
        if len(arr) > 0:
            ax1.hist(arr, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            ax1.axvline(np.mean(arr), color='red', linestyle='--', label=f'Mean: {np.mean(arr):.2f}s')
            ax1.axvline(np.median(arr), color='orange', linestyle='--', label=f'Median: {np.median(arr):.2f}s')
            ax1.set_xlabel('Travel Time (s)')
            ax1.set_ylabel('Frequency')
            ax1.set_title(f'Distribusi Waktu Tempuh P (n={len(arr)})')
            ax1.legend()
            ax1.grid(True, alpha=0.3)
            
            stats_text = f"Mean: {np.mean(arr):.2f}s\nStd: {np.std(arr):.2f}s\nMin: {np.min(arr):.2f}s\nMax: {np.max(arr):.2f}s"
            ax1.text(0.95, 0.95, stats_text, transform=ax1.transAxes, 
                     verticalalignment='top', horizontalalignment='right',
                     bbox=dict(boxstyle='round', facecolor='white', alpha=0.8), fontsize=9)
        else:
            ax1.text(0.5, 0.5, 'Tidak ada data travel time valid', ha='center', va='center')
    
    # B2. Boxplot per Network (dengan data yang sinkron)
    ax2 = fig_dash.add_subplot(gs_dash[0, 1])
    if networks and travel_times:
        # Pastikan panjang sama
        min_len = min(len(networks), len(travel_times))
        df_net = pd.DataFrame({
            'network': networks[:min_len], 
            'travel_time': travel_times[:min_len]
        })
        # Filter network dengan > 5 data
        net_counts = df_net['network'].value_counts()
        valid_nets = net_counts[net_counts > 5].index
        df_filtered = df_net[df_net['network'].isin(valid_nets)]
        if len(df_filtered) > 0:
            df_filtered.boxplot(column='travel_time', by='network', ax=ax2)
            ax2.set_title('Distribusi Travel Time per Network')
            ax2.set_xlabel('Network')
            ax2.set_ylabel('Travel Time (s)')
            ax2.grid(True, alpha=0.3)
            ax2.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='Origin')
        else:
            ax2.text(0.5, 0.5, 'Tidak cukup data per network', ha='center', va='center')
    else:
        ax2.text(0.5, 0.5, 'Tidak ada data travel time valid untuk boxplot', ha='center', va='center')
    
    # B3. Scatter Magnitude vs Travel Time
    ax3 = fig_dash.add_subplot(gs_dash[0, 2])
    if magnitudes and travel_times:
        min_len = min(len(magnitudes), len(travel_times))
        mag_arr = np.array(magnitudes[:min_len])
        tt_arr = np.array(travel_times[:min_len])
        # Filter valid
        valid_idx = ~np.isnan(mag_arr) & ~np.isnan(tt_arr) & np.isfinite(mag_arr) & np.isfinite(tt_arr)
        if np.sum(valid_idx) > 0:
            ax3.scatter(mag_arr[valid_idx], tt_arr[valid_idx], alpha=0.5, s=20, c='steelblue')
            ax3.set_xlabel('Magnitude')
            ax3.set_ylabel('Travel Time (s)')
            ax3.set_title('Magnitude vs Travel Time')
            ax3.grid(True, alpha=0.3)
            corr = np.corrcoef(mag_arr[valid_idx], tt_arr[valid_idx])[0,1]
            ax3.text(0.05, 0.95, f'Correlation: {corr:.3f}', transform=ax3.transAxes,
                     verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        else:
            ax3.text(0.5, 0.5, 'Tidak ada data valid untuk scatter', ha='center', va='center')
    
    # B4. Ringkasan Metrik
    ax4 = fig_dash.add_subplot(gs_dash[1, :])
    ax4.axis('off')
    
    # Hitung negative travel time
    neg_count = np.sum(np.array(travel_times) < 0) if travel_times else 0
    
    summary = [
        ['Total Event', str(len(all_records))],
        ['Mean Travel Time', f"{np.mean(travel_times):.2f} s" if travel_times else 'N/A'],
        ['Std Travel Time', f"{np.std(travel_times):.2f} s" if travel_times else 'N/A'],
        ['Median Travel Time', f"{np.median(travel_times):.2f} s" if travel_times else 'N/A'],
        ['Min Travel Time', f"{np.min(travel_times):.2f} s" if travel_times else 'N/A'],
        ['Max Travel Time', f"{np.max(travel_times):.2f} s" if travel_times else 'N/A'],
        ['Events with Negatif TT', str(neg_count)],
        ['Mean STA/LTA Peak', f"{np.mean(sta_lta_max):.3f}" if sta_lta_max else 'N/A'],
        ['Unique Networks', str(len(set(networks)))],
    ]
    table = ax4.table(cellText=summary, colLabels=['Metrik', 'Nilai'],
                      cellLoc='center', loc='center',
                      colColours=['#4472C4', '#4472C4'])
    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1, 2)
    ax4.set_title('📊 RINGKASAN STATISTIK VALIDASI P-WAVE', fontsize=14, pad=20)
    
    plt.tight_layout()
    dash_path = os.path.join(OUTPUT_DIR, 'p_pick_dashboard.png')
    plt.savefig(dash_path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"✅ Dashboard tersimpan: {dash_path}")
    
    # ==========================================
    # C. GRID WAVEFORM (HANYA JIKA grid_data TIDAK KOSONG)
    # ==========================================
    if len(grid_data) > 0:
        print("🖼️ Membuat grid waveform...")
        n_plot = len(grid_data)
        n_cols = 4
        n_rows = (n_plot + n_cols - 1) // n_cols
        
        fig_grid = plt.figure(figsize=(16, 4 * n_rows))
        gs_grid = GridSpec(n_rows * 2, n_cols, figure=fig_grid, hspace=0.6, wspace=0.4)
        
        for i, rec in enumerate(grid_data):
            row_major = i // n_cols
            col_major = i % n_cols
            
            # Waveform
            ax1 = fig_grid.add_subplot(gs_grid[row_major * 2, col_major])
            time_signal = np.arange(len(rec['signal'])) / SAMPLE_RATE
            time_noise = np.arange(len(rec['noise'])) / SAMPLE_RATE - len(rec['noise'])/SAMPLE_RATE
            
            ax1.plot(time_noise, rec['noise'], color='gray', alpha=0.6)
            ax1.plot(time_signal, rec['signal'], color='blue', alpha=0.8)
            ax1.axvline(x=0, color='red', linestyle='--', linewidth=1.2)
            ax1.set_xlim(-7.5, 7.5)
            ax1.set_ylim(-1.2, 1.2)
            ax1.set_title(f"{rec['key'][:20]}\nM{rec['magnitude']:.1f} | {rec['network']}.{rec['station']}", fontsize=7)
            ax1.grid(True, alpha=0.3)
            
            # STA/LTA
            ax2 = fig_grid.add_subplot(gs_grid[row_major * 2 + 1, col_major])
            time_full = np.arange(len(rec['cft'])) / SAMPLE_RATE - len(rec['noise'])/SAMPLE_RATE
            ax2.plot(time_full, rec['cft'], color='green', linewidth=1)
            ax2.axvline(x=0, color='red', linestyle='--', linewidth=1)
            ax2.set_xlim(-7.5, 7.5)
            ax2.grid(True, alpha=0.3)
            ax2.set_ylabel('STA/LTA', fontsize=6)
        
        grid_path = os.path.join(OUTPUT_DIR, 'p_pick_grid.png')
        plt.savefig(grid_path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"✅ Grid tersimpan: {grid_path}")
    else:
        print("⚠️ grid_data kosong, lewati pembuatan grid.")
    
    # ==========================================
    # D. DETAIL BEST VS WORST PICKS
    # ==========================================
    # Filter records dengan travel time valid
    valid_records = [r for r in all_records if not np.isnan(r['travel_time']) and r['travel_time'] >= 0 and r['travel_time'] < 60]
    
    if len(valid_records) >= N_BEST_WORST * 2:
        print("🔍 Membuat detail best vs worst picks...")
        best_records = sorted(valid_records, key=lambda x: x['max_cft'], reverse=True)[:N_BEST_WORST]
        worst_records = sorted(valid_records, key=lambda x: x['max_cft'])[:N_BEST_WORST]
        
        fig_detail, axes = plt.subplots(2, N_BEST_WORST, figsize=(16, 8))
        fig_detail.suptitle('Detail Waveform: Best vs Worst P-Picks', fontsize=16, fontweight='bold')
        
        for i, rec in enumerate(best_records):
            ax = axes[0, i]
            plot_single_event_detail(ax, rec['signal'], rec['noise'], rec['cft'], SAMPLE_RATE, 
                                     len(rec['noise']), f"BEST #{i+1}", rec['key'][:20], 
                                     rec['magnitude'], rec['network'], rec['station'])
            ax.set_title(f"BEST #{i+1} | STA/LTA Peak: {rec['max_cft']:.2f}", fontsize=9)
        
        for i, rec in enumerate(worst_records):
            ax = axes[1, i]
            plot_single_event_detail(ax, rec['signal'], rec['noise'], rec['cft'], SAMPLE_RATE,
                                     len(rec['noise']), f"WORST #{i+1}", rec['key'][:20],
                                     rec['magnitude'], rec['network'], rec['station'])
            ax.set_title(f"WORST #{i+1} | STA/LTA Peak: {rec['max_cft']:.2f}", fontsize=9)
        
        plt.tight_layout()
        detail_path = os.path.join(OUTPUT_DIR, 'p_pick_best_worst.png')
        plt.savefig(detail_path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"✅ Detail best/worst tersimpan: {detail_path}")
    else:
        print("⚠️ Tidak cukup data untuk best/worst picks (butuh minimal 6 data).")
    
    # ==========================================
    # E. SIMPAN STATISTIK KE CSV
    # ==========================================
    if len(all_records) > 0:
        # Buat DataFrame hanya dari data yang memiliki travel_time valid
        valid_indices = [i for i, r in enumerate(all_records) if not np.isnan(r['travel_time'])]
        if valid_indices:
            df_stats = pd.DataFrame({
                'event_id': [all_records[i]['key'] for i in valid_indices],
                'travel_time_sec': [all_records[i]['travel_time'] for i in valid_indices],
                'network': [all_records[i]['network'] for i in valid_indices],
                'station': [all_records[i]['station'] for i in valid_indices],
                'magnitude': [all_records[i]['magnitude'] for i in valid_indices],
                'sta_lta_peak': [all_records[i]['max_cft'] for i in valid_indices]
            })
            stats_path = os.path.join(OUTPUT_DIR, 'p_pick_statistics.csv')
            df_stats.to_csv(stats_path, index=False)
            print(f"✅ Statistik CSV tersimpan: {stats_path}")
        else:
            print("⚠️ Tidak ada travel time valid untuk disimpan.")
    
    # ==========================================
    # F. CETAK RINGKASAN
    # ==========================================
    print("\n" + "="*70)
    print("📊 RINGKASAN VALIDASI")
    print("="*70)
    print(f"Total event valid: {len(all_records)}")
    if travel_times:
        arr = np.array(travel_times)
        arr = arr[~np.isnan(arr) & np.isfinite(arr)]
        if len(arr) > 0:
            print(f"Mean travel time: {np.mean(arr):.3f} s")
            print(f"Median travel time: {np.median(arr):.3f} s")
            print(f"Std travel time: {np.std(arr):.3f} s")
            neg_count = np.sum(arr < 0)
            print(f"Negative travel time (picking error): {neg_count} ({neg_count/len(arr)*100:.2f}%)")
        else:
            print("Tidak ada travel time valid.")
    print(f"\n📁 Output folder: {OUTPUT_DIR}")
    print("="*70)

if __name__ == "__main__":
    validate_and_visualize(JSON_PATH)

🔍 VALIDASI P-WAVE + VISUALISASI LENGKAP (DEBUG MODE)
📂 Memuat JSON: /Volumes/Extreme SSD/json_indonesia_juli_sesi_4/extracted_data_3c_4_SYNCED.json
   Total entri: 135
   Contoh 5 key pertama: ['GE_PMBI_20240705_163113', 'GE_JAGI_20240704_113733', 'GE_MNAI_20240705_185154', 'GE_LHMI_20240705_215147', 'GE_BBJI_20240630_205201']
📊 Mengumpulkan data statistik...
🔍 Struktur sample record (key: GE_PMBI_20240705_163113):
   Keys: ['type', 'Z', 'N', 'E', 'Z_noise', 'N_noise', 'E_noise', 'metadata']
   Metadata keys: ['network', 'station', 'p_arrival', 'file', 'origin_time', 'latitude', 'longitude', 'magnitude']
   'type': se
   'Z' length: 700
   ✅ Total data terekstrak: 135
   ✅ Grid data: 48 (dari 48 yang diminta)
📈 Membuat dashboard statistik...
✅ Dashboard tersimpan: /Volumes/Extreme SSD/mcu_quake_output_replikasi_demo/validation_p_pick/p_pick_dashboard.png
🖼️ Membuat grid waveform...
✅ Grid tersimpan: /Volumes/Extreme SSD/mcu_quake_output_replikasi_demo/validation_p_pick/p_pick_grid.png


In [22]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
VALIDASI KUALITAS STA/LTA PICKING
- Membaca JSON hasil ekstraksi
- Menghitung metrik validasi
- Memberikan skor kelulusan
- Visualisasi dashboard
- Opsional: perbandingan dengan ar_pick
"""

import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from obspy.signal.trigger import recursive_sta_lta, ar_pick
import os
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# =============================================
# 1. KONFIGURASI
# =============================================
JSON_PATH = '/Volumes/Extreme SSD/json_indonesia_juli_sesi_4/extracted_data_3c_4.json'
OUTPUT_DIR = '/Volumes/Extreme SSD/mcu_quake_output_replikasi_demo/validation_stalta'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Parameter STA/LTA (sesuai preprocessing)
STA_WIN = 1.0
LTA_WIN = 10.0
TRIGGER_THRESHOLD = 3.5
SAMPLE_RATE = 100.0
TIME_BEFORE = 30.0  # detik sebelum origin (sesuai downloader)

# Ambang batas validasi
MAX_ALLOWED_NEGATIVE_PERCENT = 2.0   # maksimal 2% travel time negatif
MAX_ALLOWED_STD_TRAVEL = 30.0        # maksimal std travel time 30 detik
MIN_MEAN_TRAVEL = 10.0               # mean minimal 10 detik
MAX_MEAN_TRAVEL = 60.0               # mean maksimal 60 detik
MIN_SNR = 2.0                        # SNR minimal (untuk sinyal bagus)
MIN_STALTA_PEAK = 1.5                # puncak STA/LTA minimal

# =============================================
# 2. FUNGSI BANTUAN
# =============================================

def calculate_snr(signal, noise):
    """Hitung SNR: puncak sinyal / std noise."""
    if len(noise) == 0 or np.std(noise) < 1e-9:
        return 0
    peak_signal = np.max(np.abs(signal))
    std_noise = np.std(noise)
    return peak_signal / std_noise

def calculate_sta_lta(trace_data, sr, sta_win, lta_win):
    """Hitung STA/LTA characteristic function."""
    sta_n = int(sta_win * sr)
    lta_n = int(lta_win * sr)
    if len(trace_data) < lta_n + sta_n:
        return np.zeros(len(trace_data))
    try:
        return recursive_sta_lta(trace_data, sta_n, lta_n)
    except:
        return np.zeros(len(trace_data))

def validate_travel_time(tt):
    """Kategorikan travel time."""
    if tt < 0:
        return 'NEGATIVE'
    elif tt < 10:
        return 'VERY_CLOSE'
    elif tt < 30:
        return 'GOOD'
    elif tt < 60:
        return 'FAR'
    else:
        return 'VERY_FAR'

def get_stalta_quality(peak):
    """Kategorikan kualitas STA/LTA peak."""
    if peak < 1.0:
        return 'VERY_LOW'
    elif peak < 1.5:
        return 'LOW'
    elif peak < 3.0:
        return 'MEDIUM'
    else:
        return 'HIGH'

# =============================================
# 3. FUNGSI UTAMA VALIDASI
# =============================================

def validate_stalta(json_path):
    print("="*70)
    print("🔍 VALIDASI KUALITAS STA/LTA PICKING")
    print("="*70)
    
    # --- Load JSON ---
    if not os.path.exists(json_path):
        print(f"❌ File tidak ditemukan: {json_path}")
        return None
    
    with open(json_path, 'r') as f:
        data = json.load(f)
    print(f"📂 Total entri di JSON: {len(data)}")
    
    if len(data) == 0:
        print("❌ JSON kosong.")
        return None
    
    # --- Ekstraksi data ---
    records = []
    travel_times = []
    networks = []
    stations = []
    magnitudes = []
    stalta_peaks = []
    snrs = []
    event_ids = []
    categories = []
    qualities = []
    
    for key, record in data.items():
        try:
            if 'Z' not in record or 'Z_noise' not in record:
                continue
            
            signal = np.array(record['Z'], dtype=float)
            noise = np.array(record['Z_noise'], dtype=float)
            metadata = record.get('metadata', {})
            
            # Metadata
            p_arrival_str = metadata.get('p_arrival', '')
            origin_str = metadata.get('origin_time', '')
            mag = metadata.get('magnitude', np.nan)
            network = metadata.get('network', 'UNK')
            station = metadata.get('station', 'UNK')
            
            # Travel time
            if p_arrival_str and origin_str:
                try:
                    p_time = datetime.fromisoformat(p_arrival_str.replace('Z', '+00:00'))
                    origin_time = datetime.fromisoformat(origin_str.replace('Z', '+00:00'))
                    travel_time = (p_time - origin_time).total_seconds()
                except:
                    travel_time = np.nan
            else:
                travel_time = np.nan
            
            # SNR
            snr = calculate_snr(signal, noise)
            
            # STA/LTA peak
            full_trace = np.concatenate([noise, signal])
            cft = calculate_sta_lta(full_trace, SAMPLE_RATE, STA_WIN, LTA_WIN)
            peak = np.max(cft) if len(cft) > 0 else 0
            
            # Kategorisasi
            if not np.isnan(travel_time):
                cat = validate_travel_time(travel_time)
                qual = get_stalta_quality(peak)
            else:
                cat = 'NO_ORIGIN'
                qual = 'UNKNOWN'
            
            records.append({
                'key': key,
                'travel_time': travel_time,
                'snr': snr,
                'stalta_peak': peak,
                'network': network,
                'station': station,
                'magnitude': mag,
                'category': cat,
                'quality': qual
            })
            
            if not np.isnan(travel_time):
                travel_times.append(travel_time)
                networks.append(network)
                stations.append(station)
                magnitudes.append(mag)
                stalta_peaks.append(peak)
                snrs.append(snr)
                event_ids.append(key)
                categories.append(cat)
                qualities.append(qual)
                
        except Exception as e:
            continue
    
    total = len(records)
    valid = len(travel_times)
    print(f"✅ Data valid (dengan travel time): {valid} dari {total}")
    print(f"⚠️ Data tanpa origin_time: {total - valid}")
    
    if valid == 0:
        print("❌ Tidak ada data dengan travel time valid.")
        return None
    
    # --- Statistik ---
    tt_arr = np.array(travel_times)
    neg_count = np.sum(tt_arr < 0)
    neg_percent = neg_count / valid * 100
    mean_tt = np.mean(tt_arr)
    std_tt = np.std(tt_arr)
    median_tt = np.median(tt_arr)
    min_tt = np.min(tt_arr)
    max_tt = np.max(tt_arr)
    
    # Kategorisasi travel time
    cat_counts = {c: categories.count(c) for c in set(categories)}
    qual_counts = {q: qualities.count(q) for q in set(qualities)}
    
    # SNR statistik
    snr_arr = np.array(snrs)
    mean_snr = np.mean(snr_arr)
    std_snr = np.std(snr_arr)
    low_snr_count = np.sum(snr_arr < MIN_SNR)
    low_snr_percent = low_snr_count / valid * 100
    
    # STA/LTA peak statistik
    peak_arr = np.array(stalta_peaks)
    mean_peak = np.mean(peak_arr)
    std_peak = np.std(peak_arr)
    low_peak_count = np.sum(peak_arr < MIN_STALTA_PEAK)
    low_peak_percent = low_peak_count / valid * 100
    
    # ==========================================
    # 4. KRITERIA KELULUSAN
    # ==========================================
    print("\n" + "="*70)
    print("📊 KRITERIA VALIDASI STA/LTA")
    print("="*70)
    
    criteria = {
        'Negative travel time (%)': {
            'value': neg_percent,
            'threshold': MAX_ALLOWED_NEGATIVE_PERCENT,
            'pass': neg_percent <= MAX_ALLOWED_NEGATIVE_PERCENT,
            'unit': '%'
        },
        'Mean travel time (s)': {
            'value': mean_tt,
            'threshold': f"{MIN_MEAN_TRAVEL} - {MAX_MEAN_TRAVEL}",
            'pass': MIN_MEAN_TRAVEL <= mean_tt <= MAX_MEAN_TRAVEL,
            'unit': 's'
        },
        'Std travel time (s)': {
            'value': std_tt,
            'threshold': MAX_ALLOWED_STD_TRAVEL,
            'pass': std_tt <= MAX_ALLOWED_STD_TRAVEL,
            'unit': 's'
        },
        'Mean SNR': {
            'value': mean_snr,
            'threshold': f">= {MIN_SNR}",
            'pass': mean_snr >= MIN_SNR,
            'unit': ''
        },
        'Low SNR (<{}) (%)'.format(MIN_SNR): {
            'value': low_snr_percent,
            'threshold': '< 30%',
            'pass': low_snr_percent < 30,
            'unit': '%'
        },
        'Low STA/LTA peak (<{}) (%)'.format(MIN_STALTA_PEAK): {
            'value': low_peak_percent,
            'threshold': '< 30%',
            'pass': low_peak_percent < 30,
            'unit': '%'
        }
    }
    
    # Cetak tabel
    print(f"{'Kriteria':<30} {'Nilai':>12} {'Batas':>15} {'Status':>10}")
    print("-"*70)
    for name, info in criteria.items():
        status = "✅ PASS" if info['pass'] else "❌ FAIL"
        val_str = f"{info['value']:.2f}{info['unit']}" if info['unit'] else f"{info['value']:.2f}"
        print(f"{name:<30} {val_str:>12} {str(info['threshold']):>15} {status:>10}")
    
    # Skor keseluruhan
    total_criteria = len(criteria)
    passed_criteria = sum(1 for c in criteria.values() if c['pass'])
    overall_pass = passed_criteria == total_criteria
    
    print("\n" + "="*70)
    if overall_pass:
        print("🎉 **KESIMPULAN: STA/LTA LULUS VALIDASI**")
        print("   Semua kriteria terpenuhi. Dataset siap digunakan.")
    else:
        print("⚠️ **KESIMPULAN: STA/LTA TIDAK LULUS VALIDASI**")
        print(f"   {passed_criteria} dari {total_criteria} kriteria terpenuhi.")
        print("   Perbaiki parameter STA/LTA atau ganti dengan ar_pick.")
    print("="*70)
    
    # ==========================================
    # 5. VISUALISASI DASHBOARD
    # ==========================================
    print("\n📈 Membuat dashboard visualisasi...")
    
    fig = plt.figure(figsize=(16, 12))
    gs = GridSpec(3, 3, figure=fig, hspace=0.4, wspace=0.3)
    
    # A. Histogram Travel Time
    ax1 = fig.add_subplot(gs[0, 0])
    bins = 20
    ax1.hist(tt_arr, bins=bins, color='steelblue', edgecolor='black', alpha=0.7)
    ax1.axvline(mean_tt, color='red', linestyle='--', label=f'Mean: {mean_tt:.1f}s')
    ax1.axvline(median_tt, color='orange', linestyle='--', label=f'Median: {median_tt:.1f}s')
    ax1.set_xlabel('Travel Time (s)')
    ax1.set_ylabel('Frequency')
    ax1.set_title(f'Distribusi Travel Time (n={valid})')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # B. Kategorisasi Travel Time
    ax2 = fig.add_subplot(gs[0, 1])
    cat_labels = ['NEGATIVE', 'VERY_CLOSE', 'GOOD', 'FAR', 'VERY_FAR']
    cat_colors = ['red', 'orange', 'green', 'blue', 'purple']
    counts = [cat_counts.get(c, 0) for c in cat_labels]
    bars = ax2.bar(cat_labels, counts, color=cat_colors, alpha=0.7)
    ax2.set_xlabel('Category')
    ax2.set_ylabel('Count')
    ax2.set_title('Kategorisasi Travel Time')
    ax2.grid(True, axis='y', alpha=0.3)
    for bar, count in zip(bars, counts):
        if count > 0:
            ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, str(count), ha='center')
    
    # C. Boxplot per Network
    ax3 = fig.add_subplot(gs[0, 2])
    if len(set(networks)) > 1:
        df_net = pd.DataFrame({'network': networks, 'travel_time': travel_times})
        net_counts = df_net['network'].value_counts()
        valid_nets = net_counts[net_counts > 3].index
        df_filtered = df_net[df_net['network'].isin(valid_nets)]
        if len(df_filtered) > 0:
            df_filtered.boxplot(column='travel_time', by='network', ax=ax3)
            ax3.set_title('Travel Time per Network')
            ax3.set_xlabel('Network')
            ax3.set_ylabel('Travel Time (s)')
            ax3.grid(True, alpha=0.3)
            ax3.axhline(y=0, color='red', linestyle='--', alpha=0.5)
        else:
            ax3.text(0.5, 0.5, 'Tidak cukup data', ha='center', va='center')
    else:
        ax3.text(0.5, 0.5, f'Hanya 1 network ({networks[0]})', ha='center', va='center')
    
    # D. Scatter Magnitude vs Travel Time
    ax4 = fig.add_subplot(gs[1, 0])
    mag_arr = np.array(magnitudes)
    valid_idx = ~np.isnan(mag_arr) & np.isfinite(mag_arr)
    if np.sum(valid_idx) > 5:
        ax4.scatter(mag_arr[valid_idx], tt_arr[valid_idx], alpha=0.6, s=30, c='steelblue')
        ax4.set_xlabel('Magnitude')
        ax4.set_ylabel('Travel Time (s)')
        ax4.set_title('Magnitude vs Travel Time')
        ax4.grid(True, alpha=0.3)
        corr = np.corrcoef(mag_arr[valid_idx], tt_arr[valid_idx])[0,1]
        ax4.text(0.05, 0.95, f'Corr: {corr:.3f}', transform=ax4.transAxes, 
                 verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    else:
        ax4.text(0.5, 0.5, 'Data tidak cukup', ha='center', va='center')
    
    # E. STA/LTA Peak vs Travel Time
    ax5 = fig.add_subplot(gs[1, 1])
    ax5.scatter(tt_arr, peak_arr, alpha=0.6, s=30, c='green')
    ax5.axhline(y=MIN_STALTA_PEAK, color='red', linestyle='--', label=f'Threshold ({MIN_STALTA_PEAK})')
    ax5.set_xlabel('Travel Time (s)')
    ax5.set_ylabel('STA/LTA Peak')
    ax5.set_title('STA/LTA Peak vs Travel Time')
    ax5.grid(True, alpha=0.3)
    ax5.legend()
    
    # F. Histogram SNR
    ax6 = fig.add_subplot(gs[1, 2])
    ax6.hist(snr_arr, bins=20, color='purple', edgecolor='black', alpha=0.7)
    ax6.axvline(MIN_SNR, color='red', linestyle='--', label=f'Threshold ({MIN_SNR})')
    ax6.set_xlabel('SNR')
    ax6.set_ylabel('Frequency')
    ax6.set_title(f'Distribusi SNR (mean={mean_snr:.2f})')
    ax6.legend()
    ax6.grid(True, alpha=0.3)
    
    # G. Summary Table
    ax7 = fig.add_subplot(gs[2, :])
    ax7.axis('off')
    summary = [
        ['Total Data', str(total)],
        ['Valid Travel Time', str(valid)],
        ['Negative TT', f"{neg_count} ({neg_percent:.1f}%)"],
        ['Mean TT (s)', f"{mean_tt:.2f}"],
        ['Std TT (s)', f"{std_tt:.2f}"],
        ['Median TT (s)', f"{median_tt:.2f}"],
        ['Min TT (s)', f"{min_tt:.2f}"],
        ['Max TT (s)', f"{max_tt:.2f}"],
        ['Mean SNR', f"{mean_snr:.2f}"],
        ['Mean STA/LTA Peak', f"{mean_peak:.2f}"],
        ['Overall Status', '✅ PASS' if overall_pass else '❌ FAIL']
    ]
    table = ax7.table(cellText=summary, colLabels=['Metrik', 'Nilai'],
                      cellLoc='center', loc='center',
                      colColours=['#4472C4', '#4472C4'])
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1, 2)
    ax7.set_title('📊 RINGKASAN VALIDASI STA/LTA', fontsize=14, pad=20)
    
    plt.tight_layout()
    dashboard_path = os.path.join(OUTPUT_DIR, 'stalta_validation_dashboard.png')
    plt.savefig(dashboard_path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"✅ Dashboard tersimpan: {dashboard_path}")
    
    # ==========================================
    # 6. SIMPAN STATISTIK CSV
    # ==========================================
    df_stats = pd.DataFrame({
        'event_id': event_ids,
        'travel_time_sec': travel_times,
        'network': networks,
        'station': stations,
        'magnitude': magnitudes,
        'snr': snrs,
        'stalta_peak': stalta_peaks,
        'category': categories,
        'quality': qualities
    })
    csv_path = os.path.join(OUTPUT_DIR, 'stalta_validation_stats.csv')
    df_stats.to_csv(csv_path, index=False)
    print(f"✅ Statistik CSV tersimpan: {csv_path}")
    
    # ==========================================
    # 7. REKOMENDASI
    # ==========================================
    print("\n" + "="*70)
    print("💡 REKOMENDASI")
    print("="*70)
    
    if neg_percent > MAX_ALLOWED_NEGATIVE_PERCENT:
        print("🔴 1. Terlalu banyak travel time negatif ({:.1f}%).".format(neg_percent))
        print("   → Perbaiki fallback di pick_p_arrival: cari puncak di sekitar origin time.")
        print("   → Atau turunkan TRIGGER_THRESHOLD (misal ke 2.5) agar lebih sensitif.")
    
    if std_tt > MAX_ALLOWED_STD_TRAVEL:
        print("🔴 2. Standar deviasi travel time terlalu tinggi ({:.1f} s).".format(std_tt))
        print("   → Variasi jarak stasiun terlalu besar. Pertimbangkan filter jarak episenter.")
    
    if mean_snr < MIN_SNR:
        print("🔴 3. SNR rata-rata terlalu rendah ({:.2f}).".format(mean_snr))
        print("   → Hanya gunakan event dengan SNR > 2.0 untuk dataset final.")
    
    if low_peak_percent > 30:
        print("🔴 4. Banyak STA/LTA peak rendah ({:.1f}% < {}).".format(low_peak_percent, MIN_STALTA_PEAK))
        print("   → Tingkatkan threshold STA/LTA atau gunakan ar_pick yang lebih akurat.")
    
    if overall_pass:
        print("✅ Semua kriteria terpenuhi. STA/LTA Anda BERHASIL!")
        print("   Dataset siap digunakan untuk benchmarking MCU-Quake.")
    else:
        print("⚠️ 5. STA/LTA TIDAK LULUS validasi.")
        print("   → SARAN TERBAIK: Ganti dengan ar_pick (dari ObsPy) untuk hasil optimal.")
        print("   → Contoh implementasi sudah tersedia di dokumentasi.")
    
    print("="*70)
    
    return {
        'overall_pass': overall_pass,
        'neg_percent': neg_percent,
        'mean_tt': mean_tt,
        'std_tt': std_tt,
        'mean_snr': mean_snr,
        'mean_peak': mean_peak,
        'total': total,
        'valid': valid
    }

# =============================================
# 8. EKSEKUSI
# =============================================
if __name__ == "__main__":
    result = validate_stalta(JSON_PATH)
    if result:
        print(f"\n📊 Hasil Akhir:")
        print(f"   Total data: {result['total']}")
        print(f"   Valid: {result['valid']}")
        print(f"   Negatif TT: {result['neg_percent']:.2f}%")
        print(f"   Mean TT: {result['mean_tt']:.2f} s")
        print(f"   Std TT: {result['std_tt']:.2f} s")
        print(f"   Mean SNR: {result['mean_snr']:.2f}")
        print(f"   STA/LTA Peak: {result['mean_peak']:.2f}")
        print(f"   Status: {'✅ PASS' if result['overall_pass'] else '❌ FAIL'}")

🔍 VALIDASI KUALITAS STA/LTA PICKING
📂 Total entri di JSON: 135
✅ Data valid (dengan travel time): 0 dari 135
⚠️ Data tanpa origin_time: 135
❌ Tidak ada data dengan travel time valid.


In [25]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
SINKRONISASI KATALOG INDONESIA DENGAN JSON WAVEFORM (REVISI)
- Menambahkan origin_time, latitude, longitude, magnitude ke metadata
- Menggunakan timestamp dari nama file
- Menangani format datetime ISO8601 dengan milidetik dan timezone
"""

import json
import pandas as pd
import re
import os
from datetime import datetime

# =============================================
# 1. KONFIGURASI
# =============================================
JSON_INDONESIA_PATH = JSON_PATH = '/Volumes/Extreme SSD/json_indonesia_juli_sesi_4/extracted_data_3c_4_SYNCED.json'
CATALOG_INDONESIA_PATH = "/Volumes/Extreme SSD/katalog/hybrid_catalog_filtered.csv"
JSON_OUTPUT_PATH = '/Volumes/Extreme SSD/json_indonesia_juli_sesi_4/extracted_data_3c_4_SYNCED.json'

# =============================================
# 2. FUNGSI BANTUAN
# =============================================

def extract_timestamp_from_key(key):
    """
    Ekstrak timestamp YYYYMMDD_HHMMSS dari key JSON.
    Format: NET_STA_YYYYMMDD_HHMMSS
    """
    match = re.search(r'(\d{8})_(\d{6})', key)
    if match:
        return f"{match.group(1)}_{match.group(2)}"
    match = re.search(r'(\d{14})', key)
    if match:
        ts = match.group(1)
        return f"{ts[:8]}_{ts[8:]}"
    return None

def read_catalog(csv_path):
    """Baca katalog dengan parsing datetime robust."""
    df = pd.read_csv(csv_path)
    print(f"📂 Katalog dimuat: {len(df)} baris")
    print(f"📋 Kolom: {df.columns.tolist()}")
    
    # Deteksi kolom
    time_col = None
    for col in df.columns:
        if 'time' in col.lower() or 'datetime' in col.lower() or 'origin' in col.lower():
            time_col = col
            break
    if time_col is None:
        raise ValueError("❌ Kolom waktu tidak ditemukan.")
    print(f"🕒 Kolom waktu: '{time_col}'")
    
    lat_col = next((col for col in df.columns if 'lat' in col.lower()), None)
    lon_col = next((col for col in df.columns if 'lon' in col.lower()), None)
    mag_col = next((col for col in df.columns if 'mag' in col.lower()), None)
    
    print(f"🌐 Kolom lat: '{lat_col}', lon: '{lon_col}'")
    if mag_col:
        print(f"📏 Kolom magnitude: '{mag_col}'")
    
    # ===== PERBAIKAN: Parsing datetime =====
    # Coba dengan format ISO8601 (mengandung milidetik dan timezone)
    try:
        df['datetime'] = pd.to_datetime(df[time_col], utc=True, format='ISO8601')
        print("✅ Parsing datetime dengan format ISO8601 berhasil.")
    except Exception as e:
        print(f"⚠️ Format ISO8601 gagal: {e}")
        print("   Mencoba dengan infer_datetime_format=True...")
        try:
            df['datetime'] = pd.to_datetime(df[time_col], utc=True, infer_datetime_format=True)
            print("✅ Parsing datetime dengan infer berhasil.")
        except Exception as e2:
            print(f"❌ Gagal memparse datetime: {e2}")
            raise
    
    # Buang baris yang tidak terkonversi
    initial_len = len(df)
    df = df.dropna(subset=['datetime'])
    if len(df) < initial_len:
        print(f"⚠️ {initial_len - len(df)} baris gagal diparse dan di-drop.")
    
    # Buat timestamp key
    df['timestamp_key'] = df['datetime'].dt.strftime("%Y%m%d_%H%M%S")
    
    # Hapus duplikat timestamp (pertahankan yang pertama)
    dup_count = df.duplicated(subset=['timestamp_key']).sum()
    if dup_count > 0:
        print(f"⚠️ Ditemukan {dup_count} duplikat timestamp. Hanya yang pertama dipertahankan.")
        df = df.drop_duplicates(subset=['timestamp_key'], keep='first')
    
    print(f"✅ Total event setelah parsing dan deduplikasi: {len(df)}")
    
    return df, lat_col, lon_col, mag_col

def sync_catalog(json_path, df_catalog, lat_col, lon_col, mag_col, output_path):
    """Sinkronkan JSON dengan katalog."""
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    print(f"📂 Total entri di JSON: {len(data)}")
    
    # Buat dictionary katalog
    catalog_dict = {}
    for _, row in df_catalog.iterrows():
        key = row['timestamp_key']
        catalog_dict[key] = {
            'origin_time': row['datetime'].isoformat(),
            'latitude': row[lat_col],
            'longitude': row[lon_col],
            'magnitude': row[mag_col] if mag_col else None
        }
    
    print(f"📂 Total timestamp unik di katalog: {len(catalog_dict)}")
    
    # Sinkronisasi
    matched = 0
    unmatched = 0
    updated_json = {}
    
    for key, record in data.items():
        ts = extract_timestamp_from_key(key)
        if ts is None:
            unmatched += 1
            updated_json[key] = record
            continue
        
        if ts in catalog_dict:
            matched += 1
            cat_info = catalog_dict[ts]
            # Tambahkan metadata
            if 'metadata' not in record:
                record['metadata'] = {}
            record['metadata']['origin_time'] = cat_info['origin_time']
            record['metadata']['latitude'] = cat_info['latitude']
            record['metadata']['longitude'] = cat_info['longitude']
            if cat_info['magnitude'] is not None:
                record['metadata']['magnitude'] = float(cat_info['magnitude'])
            updated_json[key] = record
        else:
            unmatched += 1
            updated_json[key] = record
    
    print(f"✅ Match: {matched}")
    print(f"❌ Unmatch: {unmatched}")
    
    # Simpan
    with open(output_path, 'w') as f:
        json.dump(updated_json, f, indent=2)
    
    return matched, unmatched

# =============================================
# 3. MAIN
# =============================================

if __name__ == "__main__":
    print("="*70)
    print("🔗 SINKRONISASI KATALOG INDONESIA (REVISI)")
    print("="*70)
    
    # Validasi path
    if not os.path.exists(JSON_INDONESIA_PATH):
        print(f"❌ JSON tidak ditemukan: {JSON_INDONESIA_PATH}")
        exit(1)
    
    if not os.path.exists(CATALOG_INDONESIA_PATH):
        print(f"❌ Katalog tidak ditemukan: {CATALOG_INDONESIA_PATH}")
        exit(1)
    
    # Baca katalog
    try:
        df, lat_col, lon_col, mag_col = read_catalog(CATALOG_INDONESIA_PATH)
    except Exception as e:
        print(f"❌ Gagal membaca katalog: {e}")
        exit(1)
    
    # Sinkronisasi
    matched, unmatched = sync_catalog(
        JSON_INDONESIA_PATH,
        df,
        lat_col,
        lon_col,
        mag_col,
        JSON_OUTPUT_PATH
    )
    
    print("\n" + "="*70)
    print("✅ SINKRONISASI SELESAI!")
    print(f"📂 Output: {JSON_OUTPUT_PATH}")
    print(f"   Match: {matched}")
    print(f"   Unmatch: {unmatched}")
    print("="*70)

🔗 SINKRONISASI KATALOG INDONESIA (REVISI)
📂 Katalog dimuat: 49842 baris
📋 Kolom: ['datetime', 'latitude', 'longitude', 'magnitude', 'depth_km', 'source', 'event_id', 'year']
🕒 Kolom waktu: 'datetime'
🌐 Kolom lat: 'latitude', lon: 'longitude'
📏 Kolom magnitude: 'magnitude'
✅ Parsing datetime dengan format ISO8601 berhasil.
⚠️ Ditemukan 1611 duplikat timestamp. Hanya yang pertama dipertahankan.
✅ Total event setelah parsing dan deduplikasi: 48231
📂 Total entri di JSON: 135
📂 Total timestamp unik di katalog: 48231
✅ Match: 135
❌ Unmatch: 0

✅ SINKRONISASI SELESAI!
📂 Output: /Volumes/Extreme SSD/json_indonesia_juli_sesi_4/extracted_data_3c_4_SYNCED.json
   Match: 135
   Unmatch: 0
